# Week 7 — SPY Time-Series Analysis & GARCH

**Objective:** Analyze SPY daily returns for stationarity, autocorrelation, return predictability, volatility clustering, and conditional volatility using AR(1) and GARCH(1,1) models.

**Data:** SPY, 2010–2025  
**Focus:** Statistical modeling and forecasting rather than developing a trading strategy.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_squared_error
from arch import arch_model

## 2. Download SPY Data

In [ ]:
spy = yf.download(
    "SPY",
    start="2010-01-01",
    end="2026-01-01",
    auto_adjust=True
)

spy.info()

## 3. Clean the Data

In [ ]:
spy_clean = spy.xs("SPY", level="Ticker", axis=1).copy()
spy_clean.head()

## 4. Calculate Daily Returns

Simple daily return:

\[
R_t = \frac{P_t-P_{t-1}}{P_{t-1}}
\]

In [ ]:
spy_clean["SPY"] = spy_clean["Close"].pct_change()
spy_clean = spy_clean.dropna(subset=["SPY"])
spy_clean.head()

## 5. Price and Return Visualization

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(spy_clean.index, spy_clean["Close"])
plt.title("SPY Adjusted Closing Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(spy_clean.index, spy_clean["SPY"])
plt.title("SPY Daily Returns")
plt.xlabel("Date")
plt.ylabel("Return")
plt.show()

## 6. Stationarity — Augmented Dickey-Fuller Test

The null hypothesis is that the series contains a unit root. A p-value below 0.05 provides evidence against the null.

In [ ]:
adf_price = adfuller(spy_clean["Close"])
print("SPY Price")
print("ADF statistic:", adf_price[0])
print("p-value:", adf_price[1])

adf_returns = adfuller(spy_clean["SPY"])
print("\nSPY Returns")
print("ADF statistic:", adf_returns[0])
print("p-value:", adf_returns[1])

## 7. Return Autocorrelation

In [ ]:
for lag in [1, 2, 3, 5, 10, 20]:
    autocorr = spy_clean["SPY"].autocorr(lag=lag)
    print(f"Lag {lag}: {autocorr:.6f}")

## 8. ACF

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
plot_acf(spy_clean["SPY"].dropna(), lags=30, ax=ax)
plt.title("SPY Returns — Autocorrelation")
plt.show()

## 9. PACF

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
plot_pacf(spy_clean["SPY"].dropna(), lags=30, method="ywm", ax=ax)
plt.title("SPY Returns — Partial Autocorrelation")
plt.show()

## 10. Ljung-Box Test — Returns

In [ ]:
ljung_box_returns = acorr_ljungbox(
    spy_clean["SPY"].dropna(),
    lags=[1, 5, 10, 20],
    return_df=True
)
print(ljung_box_returns)

## 11. AR(1) Model

\[
R_t = c + \phi R_{t-1} + \epsilon_t
\]

In [ ]:
spy_clean["SPY_lag1"] = spy_clean["SPY"].shift(1)
ar_data = spy_clean.dropna(subset=["SPY", "SPY_lag1"])

X = sm.add_constant(ar_data["SPY_lag1"])
y = ar_data["SPY"]

ar1_model = sm.OLS(y, X).fit()
print(ar1_model.summary())

### Interpretation

A negative and statistically significant lag coefficient indicates short-term reversal. A low R² indicates weak explanatory power.

## 12. Out-of-Sample AR(1) Test

In [ ]:
split = int(len(spy_clean) * 0.8)

train = spy_clean.iloc[:split].copy()
test = spy_clean.iloc[split:].copy()

print("Training observations:", len(train))
print("Testing observations:", len(test))

In [ ]:
train["SPY_lag1"] = train["SPY"].shift(1)
test["SPY_lag1"] = test["SPY"].shift(1)

train = train.dropna(subset=["SPY", "SPY_lag1"])
test = test.dropna(subset=["SPY", "SPY_lag1"])

In [ ]:
X_train = sm.add_constant(train["SPY_lag1"])
y_train = train["SPY"]

ar1_train = sm.OLS(y_train, X_train).fit()
print(ar1_train.summary())

In [ ]:
X_test = sm.add_constant(test["SPY_lag1"])
test["prediction"] = ar1_train.predict(X_test)

mse = mean_squared_error(test["SPY"], test["prediction"])
rmse = np.sqrt(mse)
naive_rmse = np.sqrt(np.mean(test["SPY"] ** 2))

print("Test MSE:", mse)
print("Test RMSE:", rmse)
print("Naive RMSE:", naive_rmse)

## 13. Volatility Clustering

In [ ]:
spy_clean["squared_return"] = spy_clean["SPY"] ** 2

plt.figure(figsize=(12, 5))
plt.plot(spy_clean.index, spy_clean["squared_return"])
plt.title("SPY Squared Returns — Volatility Proxy")
plt.xlabel("Date")
plt.ylabel("Squared Return")
plt.show()

## 14. Autocorrelation of Squared Returns

In [ ]:
for lag in [1, 2, 3, 5, 10, 20]:
    autocorr = spy_clean["squared_return"].autocorr(lag=lag)
    print(f"Lag {lag}: {autocorr:.6f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
plot_acf(spy_clean["squared_return"].dropna(), lags=30, ax=ax)
plt.title("SPY Squared Returns — Autocorrelation")
plt.show()

## 15. Ljung-Box Test — Squared Returns

In [ ]:
ljung_box_vol = acorr_ljungbox(
    spy_clean["squared_return"].dropna(),
    lags=[1, 5, 10, 20],
    return_df=True
)
print(ljung_box_vol)

## 16. GARCH(1,1)

\[
\sigma_t^2 = \omega + \alpha\epsilon_{t-1}^2 + \beta\sigma_{t-1}^2
\]

- **ω:** baseline variance component
- **α:** response to a new shock
- **β:** persistence of previous volatility

In [ ]:
returns = spy_clean["SPY"].dropna() * 100

garch_model = arch_model(
    returns,
    mean="Constant",
    vol="GARCH",
    p=1,
    q=1
)

garch_result = garch_model.fit(disp="off")
print(garch_result.summary())

## 17. Extract GARCH Parameters

In [ ]:
mu = garch_result.params["mu"]
omega = garch_result.params["omega"]
alpha = garch_result.params["alpha[1]"]
beta = garch_result.params["beta[1]"]
persistence = alpha + beta

print("Mu:", mu)
print("Omega:", omega)
print("Alpha:", alpha)
print("Beta:", beta)
print("Alpha + Beta:", persistence)

### Interpretation

The key persistence measure is α + β. A value close to 1 indicates that volatility shocks decay slowly over time.

## 18. Conditional Volatility

In [ ]:
conditional_volatility = garch_result.conditional_volatility
print(conditional_volatility.head())

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(conditional_volatility.index, conditional_volatility)
plt.title("SPY GARCH(1,1) Conditional Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility (%)")
plt.show()

## 19. One-Day Volatility Forecast

In [ ]:
forecast = garch_result.forecast(horizon=1)

variance_forecast = forecast.variance.iloc[-1, 0]
volatility_forecast = np.sqrt(variance_forecast)

print("Forecasted variance:", variance_forecast)
print("Forecasted daily volatility:", volatility_forecast, "%")

## 20. Ten-Day Volatility Forecast

In [ ]:
forecast_10 = garch_result.forecast(horizon=10)

volatility_10 = np.sqrt(forecast_10.variance.iloc[-1])

print("10-day volatility forecast:")
print(volatility_10)

## 21. Annualized One-Day Volatility Forecast

In [ ]:
annualized_vol = volatility_forecast * np.sqrt(252)
print("Annualized volatility:", annualized_vol, "%")

## 22. Results Summary

In [ ]:
results_summary = pd.DataFrame({
    "Metric": [
        "AR(1) coefficient",
        "AR(1) R²",
        "AR(1) test RMSE",
        "Naive RMSE",
        "GARCH omega",
        "GARCH alpha",
        "GARCH beta",
        "GARCH alpha + beta"
    ],
    "Value": [
        ar1_train.params["SPY_lag1"],
        ar1_train.rsquared,
        rmse,
        naive_rmse,
        omega,
        alpha,
        beta,
        persistence
    ]
})

results_summary

# Conclusion

This project examined the time-series properties of SPY daily returns from 2010 to 2025.

The analysis tested stationarity, autocorrelation, return predictability, volatility clustering, and conditional volatility. An AR(1) model found statistically significant short-term negative autocorrelation, but its explanatory and out-of-sample predictive power was weak.

Squared returns displayed persistent dependence, providing evidence of volatility clustering. A GARCH(1,1) model was then used to estimate time-varying conditional volatility.

The overall result demonstrates an important distinction in financial time-series modeling: daily return direction is difficult to predict using simple autoregressive information, while volatility exhibits substantially stronger temporal structure and can be modeled using GARCH.